# TrashCan 1.0 — EDA
Runs the real validation/EDA pipeline (`src/data/validate_annotations.py`) against the downloaded dataset and displays the figures. Nothing here is fabricated — every number comes from parsing the actual COCO json + reading the actual image files under `data/raw/trashcan/`.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
from src.data.validate_annotations import validate_full_dataset, _print_report
from src.utils.visualization import plot_class_distribution, plot_image_size_distribution, plot_objects_per_image
from src.config import FIGURES_DIR

report, train_coco, val_coco = validate_full_dataset(compute_hashes=True)
_print_report(report)

In [ ]:
plot_class_distribution(dict(report.class_counts), FIGURES_DIR / 'class_distribution.png');

In [ ]:
plot_image_size_distribution({f'{w}x{h}': c for (w,h), c in report.image_sizes.items()}, FIGURES_DIR / 'image_size_distribution.png');

In [ ]:
plot_objects_per_image(dict(report.objects_per_image), FIGURES_DIR / 'objects_per_image.png');

## Sample images with ground-truth annotations
Draws real polygon masks from the COCO annotations onto a few train images, to sanity-check the label conversion visually before training.

In [ ]:
import cv2, numpy as np
from src.config import RAW_TRAIN_IMAGES, CLASS_NAMES
from src.utils.visualization import draw_detections, plot_sample_grid

cat_id_to_name = {c['id']: c['name'] for c in train_coco['categories']}
anns_by_img = {}
for a in train_coco['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)

samples = []
for img in train_coco['images'][:8]:
    path = RAW_TRAIN_IMAGES / img['file_name']
    if not path.exists():
        continue
    im = cv2.imread(str(path))
    anns = anns_by_img.get(img['id'], [])
    boxes = np.array([[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in anns]) if anns else np.zeros((0,4))
    cls_ids = np.array([CLASS_NAMES.index(cat_id_to_name[a['category_id']]) for a in anns])
    confs = np.ones(len(anns))
    annotated = draw_detections(im, boxes, cls_ids, confs, CLASS_NAMES) if len(anns) else im
    samples.append((annotated, img['file_name']))

plot_sample_grid(samples, FIGURES_DIR / 'ground_truth_samples.png', title='Ground truth samples');